# 04. Статистика и машинное обучение для ЭЭГ-признаков при ЧМТ

## Назначение ноутбука

Данный ноутбук объединяет признаки, рассчитанные в предыдущих ноутбуках, выполняет аудит матрицы признаков, статистическое сравнение групп и построение моделей машинного обучения.

Входные файлы:

```text
spectral_features_record_level.csv
temporal_connectivity_event_features_record_level.csv
```

Основные задачи ноутбука:

1. объединить спектральные, временные, сетевые и событийные признаки;
2. провести аудит признакового пространства;
3. сформировать clean feature sets;
4. выполнить record-level и subject-level EDA;
5. провести межгрупповую статистику;
6. построить baseline-модель на спектральных признаках;
7. построить модель на всех признаках;
8. построить компактную событийную модель;
9. выполнить permutation test;
10. провести stress tests;
11. сохранить итоговые таблицы и рисунки для ВКР.

Главный выходной файл:

```text
all_features_record_level_core.csv
```

## Связь с предыдущими ноутбуками

```text
01_prepare_eeg_dataset.ipynb
        ↓
analysis_ready_inventory_cleaned.csv

02_spectral_features.ipynb
        ↓
spectral_features_record_level.csv

03_temporal_connectivity_events.ipynb
        ↓
temporal_connectivity_event_features_record_level.csv

04_statistics_and_ml_v2.ipynb
        ↓
all_features_record_level_core.csv
model_comparison_summary_for_thesis.csv
```

Во всех моделях используется защита от утечки данных: записи одного субъекта не должны одновременно попадать в обучающую и тестовую части.

In [ ]:
# @title 0.1. Установка зависимостей

!pip -q install pandas numpy scipy matplotlib scikit-learn statsmodels tqdm

In [ ]:
# @title 0.2. Импорты, стиль графиков и базовые настройки

import json
import logging
import warnings
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    GroupShuffleSplit,
    StratifiedGroupKFold,
    permutation_test_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests
from tqdm import tqdm

warnings.filterwarnings("ignore")

# Единые цвета групп во всех ноутбуках проекта.
COL_TBI = "#9B1B30"
COL_CONTROL = "#1F3F7A"

GROUP_COLORS = {
    "TBI": COL_TBI,
    "Control": COL_CONTROL,
}

GROUP_LABELS_RU = {
    "TBI": "ЧМТ",
    "Control": "Контроль",
}

plt.rcParams.update(
    {
        "figure.figsize": (12, 7),
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.family": "serif",
        "font.size": 12,
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "grid.linestyle": "--",
    }
)


def save_figure(fig, output_path: Path) -> None:
    """
    Сохраняет рисунок с аккуратными отступами.

    Parameters
    ----------
    fig : matplotlib.figure.Figure
        Объект рисунка.
    output_path : Path
        Путь для сохранения изображения.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, bbox_inches="tight", dpi=300)
    print(f"Рисунок сохранён: {output_path}")


def save_table(df: pd.DataFrame, output_path: Path) -> None:
    """
    Сохраняет таблицу в CSV-файл.

    Parameters
    ----------
    df : pd.DataFrame
        Таблица для сохранения.
    output_path : Path
        Путь для сохранения CSV-файла.
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Таблица сохранена: {output_path}")

# 1. Конфигурация анализа

В этом разделе задаются пути к таблицам признаков, параметры отбора признаков, настройки статистики и параметры моделей.

По умолчанию используется рабочая папка второго варианта предобработки:

```text
/content/drive/MyDrive/VKR/TBI/work-ica-ar
```

In [ ]:
# @title 1.1. Конфигурация статистики и ML

CONFIG = {
    # ------------------------------------------------------------------
    # Пути
    # ------------------------------------------------------------------
    "work_dir": Path("/content/drive/MyDrive/VKR/TBI/work-ica-ar"),

    "spectral_features_path": (
        Path("/content/drive/MyDrive/VKR/TBI/work-ica-ar")
        / "output_spectral_features"
        / "tables"
        / "spectral_features_record_level.csv"
    ),

    "temporal_connectivity_event_features_path": (
        Path("/content/drive/MyDrive/VKR/TBI/work-ica-ar")
        / "output_temporal_connectivity_events"
        / "tables"
        / "temporal_connectivity_event_features_record_level.csv"
    ),

    # ------------------------------------------------------------------
    # Аудит признаков
    # ------------------------------------------------------------------
    "max_missing_soft": 0.30,
    "max_missing_strict": 0.05,
    "high_corr_threshold": 0.98,

    # ------------------------------------------------------------------
    # Разбиение и валидация
    # ------------------------------------------------------------------
    "random_state": 97,
    "test_size": 0.25,
    "n_splits_cv": 5,

    # ------------------------------------------------------------------
    # Статистика
    # ------------------------------------------------------------------
    "alpha": 0.05,

    # ------------------------------------------------------------------
    # Модели
    # ------------------------------------------------------------------
    "logreg_max_iter": 5000,
    "logreg_c": 1.0,
    "class_weight": "balanced",

    # ------------------------------------------------------------------
    # Permutation tests
    # ------------------------------------------------------------------
    "n_permutations": 500,

    # ------------------------------------------------------------------
    # Компактная событийная модель
    # ------------------------------------------------------------------
    # Если этих признаков нет, ниже автоматически будут выбраны первые
    # доступные event-признаки с высокой величиной эффекта.
    "event_compact_features": [
        "event__n_events",
        "event__event_mean_z_peak",
        "event__event_mean_active_channels",
    ],
}

OUTPUT_DIR = CONFIG["work_dir"] / "output_statistics_ml"

OUT = {
    "tables": OUTPUT_DIR / "tables",
    "figures": OUTPUT_DIR / "figures",
    "logs": OUTPUT_DIR / "logs",
    "models": OUTPUT_DIR / "models",
}

for path in OUT.values():
    path.mkdir(parents=True, exist_ok=True)

print("Конфигурация статистики и ML задана.")
print(f"Рабочая папка: {OUTPUT_DIR}")

In [ ]:
# @title 1.2. Настройка логирования

def setup_logger(output_dir: Path) -> logging.Logger:
    """
    Настраивает logger для текущего ноутбука.

    Parameters
    ----------
    output_dir : Path
        Папка для сохранения логов.

    Returns
    -------
    logging.Logger
        Настроенный logger.
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    logger = logging.getLogger("statistics_ml_v2")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    log_path = output_dir / (
        "statistics_ml_v2_"
        f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
    )

    file_handler = logging.FileHandler(log_path, encoding="utf-8")
    file_handler.setLevel(logging.INFO)

    stream_handler = logging.StreamHandler()
    stream_handler.setLevel(logging.INFO)

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    file_handler.setFormatter(formatter)
    stream_handler.setFormatter(formatter)

    logger.addHandler(file_handler)
    logger.addHandler(stream_handler)

    logger.info("Логирование статистики и ML запущено.")
    logger.info("Файл лога: %s", log_path)

    return logger


logger = setup_logger(OUT["logs"])

In [ ]:
# @title 1.3. Проверка входных файлов

required_files = {
    "spectral_features": CONFIG["spectral_features_path"],
    "temporal_connectivity_event_features": (
        CONFIG["temporal_connectivity_event_features_path"]
    ),
}

print("Проверка входных файлов")
print("-" * 70)

for name, path in required_files.items():
    status = "OK" if path.exists() else "НЕ НАЙДЕН"
    print(f"{name}: {status}")
    print(f"  {path}")

missing_files = [
    path for path in required_files.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Часть входных файлов не найдена. "
        "Сначала выполните ноутбуки 02 и 03."
    )

logger.info("Все входные файлы найдены.")

# 2. Загрузка и объединение признаков

На этом этапе загружаются таблицы признаков из ноутбуков 02 и 03.

Объединение выполняется по ключам:

```text
group
subject_id
record_id
```

После объединения формируется основная record-level таблица признаков.

In [ ]:
# @title 2.1. Загрузка таблиц признаков

spectral_features = pd.read_csv(CONFIG["spectral_features_path"])
temporal_event_features = pd.read_csv(
    CONFIG["temporal_connectivity_event_features_path"]
)

id_columns = ["group", "subject_id", "record_id"]

for name, df in {
    "spectral_features": spectral_features,
    "temporal_event_features": temporal_event_features,
}.items():
    missing_columns = [
        column for column in id_columns
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"В таблице {name} отсутствуют колонки: {missing_columns}"
        )

print("Спектральные признаки:", spectral_features.shape)
print("Временные/сетевые/событийные признаки:", temporal_event_features.shape)

display(spectral_features.head())
display(temporal_event_features.head())

In [ ]:
# @title 2.2. Объединение признаков

all_features_record_level = spectral_features.merge(
    temporal_event_features,
    on=id_columns,
    how="outer",
    validate="one_to_one",
)

all_features_path = OUT["tables"] / "all_features_record_level_core.csv"
save_table(all_features_record_level, all_features_path)

print("Объединённая таблица признаков:")
print(all_features_record_level.shape)

display(
    all_features_record_level
    .groupby("group")
    .agg(
        n_subjects=("subject_id", "nunique"),
        n_records=("record_id", "count"),
    )
    .reset_index()
)

In [ ]:
# @title 2.3. Карта блоков признаков

feature_columns = [
    column for column in all_features_record_level.columns
    if column not in id_columns
]

feature_map = pd.DataFrame(
    {
        "feature": feature_columns,
        "block": [
            feature.split("__")[0]
            if "__" in feature else "unknown"
            for feature in feature_columns
        ],
    }
)

feature_map_path = OUT["tables"] / "all_features_record_level_feature_map.csv"
save_table(feature_map, feature_map_path)

display(
    feature_map
    .groupby("block")
    .agg(n_features=("feature", "count"))
    .reset_index()
    .sort_values("n_features", ascending=False)
)

# 3. Аудит матрицы признаков

Перед статистикой и ML проверяются:

1. доля пропусков;
2. признаки с нулевой дисперсией;
3. признаки с высокой корреляцией;
4. число признаков по блокам.

На основе аудита формируются несколько наборов признаков: мягкий, строгий и финальный.

In [ ]:
# @title 3.1. Missingness, zero variance и типы признаков

numeric_feature_columns = [
    column for column in feature_columns
    if pd.api.types.is_numeric_dtype(all_features_record_level[column])
]

feature_audit = pd.DataFrame(
    {
        "feature": numeric_feature_columns,
        "missing_prop": all_features_record_level[
            numeric_feature_columns
        ].isna().mean().values,
        "n_unique": [
            all_features_record_level[column].nunique(dropna=True)
            for column in numeric_feature_columns
        ],
    }
)

feature_audit["zero_variance"] = feature_audit["n_unique"] <= 1
feature_audit["block"] = feature_audit["feature"].map(
    feature_map.set_index("feature")["block"]
)

feature_audit["keep_soft"] = (
    (feature_audit["missing_prop"] <= CONFIG["max_missing_soft"])
    & (~feature_audit["zero_variance"])
)

feature_audit["keep_strict"] = (
    (feature_audit["missing_prop"] <= CONFIG["max_missing_strict"])
    & (~feature_audit["zero_variance"])
)

feature_audit_path = OUT["tables"] / "feature_audit.csv"
save_table(feature_audit, feature_audit_path)

print("Число числовых признаков:", len(numeric_feature_columns))
print("Средняя доля пропусков:", feature_audit["missing_prop"].mean())
print("Признаки с нулевой дисперсией:", int(feature_audit["zero_variance"].sum()))
print("Soft features:", int(feature_audit["keep_soft"].sum()))
print("Strict features:", int(feature_audit["keep_strict"].sum()))

display(
    feature_audit
    .sort_values(["missing_prop", "zero_variance"], ascending=False)
    .head(30)
)

In [ ]:
# @title 3.2. Поиск высококоррелированных признаков

def find_highly_correlated_features(
    df: pd.DataFrame,
    features: list[str],
    threshold: float,
) -> list[str]:
    """
    Находит признаки, которые можно удалить из-за высокой корреляции.

    Parameters
    ----------
    df : pd.DataFrame
        Таблица признаков.
    features : list[str]
        Список признаков.
    threshold : float
        Порог абсолютной корреляции.

    Returns
    -------
    list[str]
        Список признаков для удаления.
    """
    if len(features) < 2:
        return []

    feature_df = df[features].copy()
    feature_df = feature_df.fillna(feature_df.median(numeric_only=True))

    corr = feature_df.corr().abs()
    upper = corr.where(
        np.triu(np.ones(corr.shape), k=1).astype(bool)
    )

    to_drop = [
        column for column in upper.columns
        if any(upper[column] > threshold)
    ]

    return to_drop


soft_features_initial = feature_audit.loc[
    feature_audit["keep_soft"],
    "feature",
].tolist()

strict_features_initial = feature_audit.loc[
    feature_audit["keep_strict"],
    "feature",
].tolist()

high_corr_drop = find_highly_correlated_features(
    df=all_features_record_level,
    features=soft_features_initial,
    threshold=CONFIG["high_corr_threshold"],
)

soft_features = [
    feature for feature in soft_features_initial
    if feature not in high_corr_drop
]

strict_features = [
    feature for feature in strict_features_initial
    if feature not in high_corr_drop
]

clean_feature_sets = {
    "neuro_clean_soft": soft_features,
    "neuro_clean_strict": strict_features,
    "neuro_clean_final": soft_features,
}

high_corr_drop_df = pd.DataFrame({"feature": high_corr_drop})
save_table(high_corr_drop_df, OUT["tables"] / "high_corr_features_to_drop.csv")

feature_set_summary = pd.DataFrame(
    [
        {
            "feature_set": name,
            "n_features": len(features),
        }
        for name, features in clean_feature_sets.items()
    ]
)

save_table(
    feature_set_summary,
    OUT["tables"] / "clean_feature_set_summary.csv",
)

display(feature_set_summary)

In [ ]:
# @title 3.3. Визуализация числа признаков по блокам

block_summary = (
    feature_map
    .groupby("block")
    .agg(n_features=("feature", "count"))
    .reset_index()
    .sort_values("n_features", ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 6))

ax.barh(
    block_summary["block"],
    block_summary["n_features"],
)

ax.set_title("Число признаков по блокам")
ax.set_xlabel("Количество признаков")
ax.set_ylabel("Блок")

for _, row in block_summary.iterrows():
    ax.text(
        row["n_features"],
        row["block"],
        f" {int(row['n_features'])}",
        va="center",
    )

figure_path = OUT["figures"] / "feature_blocks_count.png"
save_figure(fig, figure_path)

plt.show()

# 4. Агрегация на уровне субъекта

Признаки изначально рассчитаны на уровне записи. Для части статистических проверок полезно получить subject-level таблицу.

Если у субъекта несколько записей, признаки агрегируются медианой.

In [ ]:
# @title 4.1. Subject-level агрегация

def aggregate_to_subject_level(
    df: pd.DataFrame,
    features: list[str],
) -> pd.DataFrame:
    """
    Агрегирует record-level признаки на уровень субъекта.

    Parameters
    ----------
    df : pd.DataFrame
        Record-level таблица.
    features : list[str]
        Список признаков.

    Returns
    -------
    pd.DataFrame
        Subject-level таблица.
    """
    subject_df = (
        df[id_columns + features]
        .groupby(["group", "subject_id"], as_index=False)
        .median(numeric_only=True)
    )

    return subject_df


subject_features = aggregate_to_subject_level(
    df=all_features_record_level,
    features=clean_feature_sets["neuro_clean_final"],
)

subject_features_path = OUT["tables"] / "all_features_subject_level_core.csv"
save_table(subject_features, subject_features_path)

print("Subject-level таблица:")
print(subject_features.shape)

display(
    subject_features
    .groupby("group")
    .agg(n_subjects=("subject_id", "nunique"))
    .reset_index()
)

# 5. Межгрупповая статистика

Для каждого признака выполняется сравнение групп ЧМТ и контроля.

Сохраняются:

- среднее в группе ЧМТ;
- среднее в контрольной группе;
- разность средних;
- Hedges' g;
- Mann–Whitney p-value;
- q-value после FDR-коррекции.

Основная статистическая таблица строится на subject-level данных.

In [ ]:
# @title 5.1. Функции статистического сравнения

def hedges_g(x: np.ndarray, y: np.ndarray) -> float:
    """
    Рассчитывает Hedges' g для двух независимых групп.

    Parameters
    ----------
    x : np.ndarray
        Значения первой группы.
    y : np.ndarray
        Значения второй группы.

    Returns
    -------
    float
        Hedges' g.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    n_x = len(x)
    n_y = len(y)

    if n_x < 2 or n_y < 2:
        return np.nan

    pooled_sd = np.sqrt(
        ((n_x - 1) * np.var(x, ddof=1) + (n_y - 1) * np.var(y, ddof=1))
        / (n_x + n_y - 2)
    )

    if pooled_sd == 0:
        return np.nan

    cohen_d = (np.mean(x) - np.mean(y)) / pooled_sd
    correction = 1 - 3 / (4 * (n_x + n_y) - 9)

    return float(cohen_d * correction)


def compare_feature_between_groups(
    df: pd.DataFrame,
    feature: str,
) -> dict:
    """
    Сравнивает группы по одному признаку.

    Parameters
    ----------
    df : pd.DataFrame
        Таблица признаков.
    feature : str
        Название признака.

    Returns
    -------
    dict
        Результаты сравнения.
    """
    tbi_values = df[df["group"] == "TBI"][feature].to_numpy(dtype=float)
    ctl_values = df[df["group"] == "Control"][feature].to_numpy(dtype=float)

    tbi_values = tbi_values[np.isfinite(tbi_values)]
    ctl_values = ctl_values[np.isfinite(ctl_values)]

    if len(tbi_values) < 2 or len(ctl_values) < 2:
        p_value = np.nan
        effect = np.nan
    else:
        _, p_value = stats.mannwhitneyu(
            tbi_values,
            ctl_values,
            alternative="two-sided",
        )
        effect = hedges_g(tbi_values, ctl_values)

    return {
        "feature": feature,
        "block": feature.split("__")[0] if "__" in feature else "unknown",
        "tbi_mean": np.nanmean(tbi_values),
        "control_mean": np.nanmean(ctl_values),
        "delta_tbi_minus_control": (
            np.nanmean(tbi_values) - np.nanmean(ctl_values)
        ),
        "hedges_g": effect,
        "p_value": p_value,
        "n_tbi": len(tbi_values),
        "n_control": len(ctl_values),
    }

In [ ]:
# @title 5.2. Subject-level статистика

stats_features = clean_feature_sets["neuro_clean_final"]

statistics_records = [
    compare_feature_between_groups(
        df=subject_features,
        feature=feature,
    )
    for feature in stats_features
]

feature_statistics_subject = pd.DataFrame(statistics_records)

feature_statistics_subject["q_fdr"] = multipletests(
    feature_statistics_subject["p_value"].fillna(1.0),
    method="fdr_bh",
)[1]

feature_statistics_subject = feature_statistics_subject.sort_values("q_fdr")

statistics_path = OUT["tables"] / "feature_statistics_subject_level.csv"
save_table(feature_statistics_subject, statistics_path)

display(feature_statistics_subject.head(30))

In [ ]:
# @title 5.3. Топ-признаки по величине эффекта

top_effects = (
    feature_statistics_subject
    .dropna(subset=["hedges_g"])
    .assign(abs_g=lambda df: df["hedges_g"].abs())
    .sort_values("abs_g", ascending=False)
    .head(20)
    .sort_values("abs_g", ascending=True)
)

fig, ax = plt.subplots(figsize=(10, 8))

ax.barh(
    top_effects["feature"],
    top_effects["hedges_g"],
)

ax.axvline(0, color="black", linewidth=1)
ax.set_title("Топ-20 признаков по |Hedges' g|")
ax.set_xlabel("Hedges' g")
ax.set_ylabel("Признак")

figure_path = OUT["figures"] / "top_features_by_hedges_g_subject_level.png"
save_figure(fig, figure_path)

plt.show()

# 6. Подготовка данных для ML

Модели обучаются на record-level таблице, но все разбиения выполняются на уровне `subject_id`.

Это исключает ситуацию, когда разные записи одного субъекта попадают одновременно в train и test.

In [ ]:
# @title 6.1. Функции подготовки данных и модели

def prepare_ml_data(
    df: pd.DataFrame,
    selected_features: list[str],
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    """
    Формирует X, y и groups для ML.

    Parameters
    ----------
    df : pd.DataFrame
        Таблица признаков.
    selected_features : list[str]
        Список признаков.

    Returns
    -------
    tuple[pd.DataFrame, np.ndarray, np.ndarray]
        X, y и groups.
    """
    model_df = df[id_columns + selected_features].copy()

    y = (model_df["group"] == "TBI").astype(int).to_numpy()
    groups = model_df["subject_id"].to_numpy()

    X = model_df[selected_features].copy()

    return X, y, groups


def make_logreg_pipeline() -> Pipeline:
    """
    Создаёт pipeline логистической регрессии.

    Returns
    -------
    Pipeline
        Imputer + scaler + logistic regression.
    """
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    C=CONFIG["logreg_c"],
                    max_iter=CONFIG["logreg_max_iter"],
                    class_weight=CONFIG["class_weight"],
                    solver="liblinear",
                    random_state=CONFIG["random_state"],
                ),
            ),
        ]
    )


def evaluate_binary_classifier(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_proba: np.ndarray,
) -> dict:
    """
    Рассчитывает метрики бинарной классификации.

    Parameters
    ----------
    y_true : np.ndarray
        Истинные метки.
    y_pred : np.ndarray
        Предсказанные метки.
    y_proba : np.ndarray
        Вероятности класса 1.

    Returns
    -------
    dict
        Метрики качества.
    """
    return {
        "roc_auc": roc_auc_score(y_true, y_proba),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }

In [ ]:
# @title 6.2. Subject-level holdout split

ml_features_final = clean_feature_sets["neuro_clean_final"]

X_all, y_all, groups_all = prepare_ml_data(
    df=all_features_record_level,
    selected_features=ml_features_final,
)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=CONFIG["test_size"],
    random_state=CONFIG["random_state"],
)

train_idx, test_idx = next(
    splitter.split(X_all, y_all, groups=groups_all)
)

print("Размер train:", len(train_idx))
print("Размер test:", len(test_idx))
print("Субъектов train:", len(np.unique(groups_all[train_idx])))
print("Субъектов test:", len(np.unique(groups_all[test_idx])))

overlap_subjects = set(groups_all[train_idx]) & set(groups_all[test_idx])
print("Пересечение субъектов train/test:", len(overlap_subjects))

if overlap_subjects:
    raise ValueError("Обнаружена утечка субъектов между train и test.")

# 7. Наборы признаков для моделей

Сравниваются несколько моделей:

1. **spectral baseline** — спектральные признаки;
2. **bp_idx baseline** — bandpower и slow/fast индексы;
3. **all features** — полный clean-набор признаков;
4. **compact event model** — компактный событийный набор.

Такое сравнение позволяет показать, насколько событийные признаки информативны относительно классических спектральных baseline.

In [ ]:
# @title 7.1. Определение feature sets

def select_features_by_prefix(
    features: list[str],
    prefixes: tuple[str, ...],
) -> list[str]:
    """
    Выбирает признаки по префиксам.

    Parameters
    ----------
    features : list[str]
        Список признаков.
    prefixes : tuple[str, ...]
        Префиксы.

    Returns
    -------
    list[str]
        Отобранные признаки.
    """
    return [
        feature for feature in features
        if feature.startswith(prefixes)
    ]


spectral_feature_set = select_features_by_prefix(
    ml_features_final,
    ("bp__", "spec_shape__", "aper_alpha__"),
)

bp_idx_feature_set = [
    feature for feature in ml_features_final
    if (
        feature.startswith("bp__")
        and (
            "rel_power" in feature
            or "abs_power" in feature
            or "idx_" in feature
        )
    )
]

event_feature_set = select_features_by_prefix(
    ml_features_final,
    ("event__",),
)

available_event_compact_features = [
    feature for feature in CONFIG["event_compact_features"]
    if feature in ml_features_final
]

if len(available_event_compact_features) == 0:
    event_stats = feature_statistics_subject[
        feature_statistics_subject["feature"].isin(event_feature_set)
    ].copy()

    event_stats["abs_g"] = event_stats["hedges_g"].abs()

    available_event_compact_features = (
        event_stats
        .sort_values("abs_g", ascending=False)
        .head(3)["feature"]
        .tolist()
    )

feature_sets = {
    "spectral_baseline": spectral_feature_set,
    "bp_idx_baseline": bp_idx_feature_set,
    "all_features": ml_features_final,
    "compact_event": available_event_compact_features,
}

feature_sets = {
    name: features
    for name, features in feature_sets.items()
    if len(features) > 0
}

feature_set_rows = [
    {
        "model": name,
        "n_features": len(features),
        "features": "|".join(features),
    }
    for name, features in feature_sets.items()
]

feature_set_table = pd.DataFrame(feature_set_rows)
save_table(feature_set_table, OUT["tables"] / "ml_feature_sets.csv")

display(feature_set_table[["model", "n_features"]])

# 8. Holdout-оценка моделей

В этом разделе модели обучаются на фиксированном subject-level holdout-разбиении.

Метрики:

- ROC-AUC;
- balanced accuracy;
- accuracy;
- precision;
- recall;
- F1-score.

In [ ]:
# @title 8.1. Функции обучения и оценки holdout

def fit_and_evaluate_holdout(
    df: pd.DataFrame,
    selected_features: list[str],
    model_name: str,
    train_idx: np.ndarray,
    test_idx: np.ndarray,
) -> tuple[dict, Pipeline]:
    """
    Обучает и оценивает модель на fixed holdout.

    Parameters
    ----------
    df : pd.DataFrame
        Таблица признаков.
    selected_features : list[str]
        Список признаков.
    model_name : str
        Название модели.
    train_idx : np.ndarray
        Индексы train.
    test_idx : np.ndarray
        Индексы test.

    Returns
    -------
    tuple[dict, Pipeline]
        Метрики и обученный pipeline.
    """
    X, y, _ = prepare_ml_data(df, selected_features)

    pipeline = make_logreg_pipeline()

    pipeline.fit(X.iloc[train_idx], y[train_idx])

    y_pred = pipeline.predict(X.iloc[test_idx])
    y_proba = pipeline.predict_proba(X.iloc[test_idx])[:, 1]

    metrics = evaluate_binary_classifier(
        y_true=y[test_idx],
        y_pred=y_pred,
        y_proba=y_proba,
    )

    metrics["model"] = model_name
    metrics["n_features"] = len(selected_features)
    metrics["n_train"] = len(train_idx)
    metrics["n_test"] = len(test_idx)

    return metrics, pipeline

In [ ]:
# @title 8.2. Holdout-сравнение моделей

model_results = []
trained_models = {}

for model_name, selected_features in feature_sets.items():
    metrics, model = fit_and_evaluate_holdout(
        df=all_features_record_level,
        selected_features=selected_features,
        model_name=model_name,
        train_idx=train_idx,
        test_idx=test_idx,
    )

    model_results.append(metrics)
    trained_models[model_name] = model

model_comparison_holdout = pd.DataFrame(model_results)
model_comparison_holdout = model_comparison_holdout.sort_values(
    "roc_auc",
    ascending=False,
)

save_table(
    model_comparison_holdout,
    OUT["tables"] / "model_comparison_holdout.csv",
)

display(model_comparison_holdout)

In [ ]:
# @title 8.3. ROC-кривые holdout

fig, ax = plt.subplots(figsize=(8, 7))

for model_name, selected_features in feature_sets.items():
    if model_name not in trained_models:
        continue

    X, y, _ = prepare_ml_data(
        all_features_record_level,
        selected_features,
    )

    y_proba = trained_models[model_name].predict_proba(
        X.iloc[test_idx]
    )[:, 1]

    fpr, tpr, _ = roc_curve(y[test_idx], y_proba)
    auc_value = roc_auc_score(y[test_idx], y_proba)

    ax.plot(
        fpr,
        tpr,
        linewidth=2,
        label=f"{model_name} (AUC={auc_value:.3f})",
    )

ax.plot([0, 1], [0, 1], linestyle="--", color="black", linewidth=1)
ax.set_title("ROC-кривые моделей на subject-level holdout")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.legend(frameon=False, loc="lower right")

figure_path = OUT["figures"] / "roc_curves_holdout.png"
save_figure(fig, figure_path)

plt.show()

# 9. Grouped cross-validation

Для устойчивой оценки используется `StratifiedGroupKFold`.

В каждом fold сохраняется баланс классов, а записи одного субъекта остаются только в одной части разбиения.

In [ ]:
# @title 9.1. Grouped CV

def cross_validate_grouped(
    df: pd.DataFrame,
    selected_features: list[str],
    model_name: str,
) -> pd.DataFrame:
    """
    Выполняет StratifiedGroupKFold оценку модели.

    Parameters
    ----------
    df : pd.DataFrame
        Таблица признаков.
    selected_features : list[str]
        Список признаков.
    model_name : str
        Название модели.

    Returns
    -------
    pd.DataFrame
        Метрики по fold.
    """
    X, y, groups = prepare_ml_data(df, selected_features)

    cv = StratifiedGroupKFold(
        n_splits=CONFIG["n_splits_cv"],
        shuffle=True,
        random_state=CONFIG["random_state"],
    )

    records = []

    for fold_index, (cv_train_idx, cv_test_idx) in enumerate(
        cv.split(X, y, groups),
        start=1,
    ):
        pipeline = make_logreg_pipeline()
        pipeline.fit(X.iloc[cv_train_idx], y[cv_train_idx])

        y_pred = pipeline.predict(X.iloc[cv_test_idx])
        y_proba = pipeline.predict_proba(X.iloc[cv_test_idx])[:, 1]

        metrics = evaluate_binary_classifier(
            y_true=y[cv_test_idx],
            y_pred=y_pred,
            y_proba=y_proba,
        )

        records.append(
            {
                "model": model_name,
                "fold": fold_index,
                "n_features": len(selected_features),
                "n_train": len(cv_train_idx),
                "n_test": len(cv_test_idx),
                **metrics,
            }
        )

    return pd.DataFrame(records)


cv_results = []

for model_name, selected_features in feature_sets.items():
    fold_df = cross_validate_grouped(
        df=all_features_record_level,
        selected_features=selected_features,
        model_name=model_name,
    )
    cv_results.append(fold_df)

cv_results = pd.concat(cv_results, ignore_index=True)

save_table(
    cv_results,
    OUT["tables"] / "model_comparison_grouped_cv_folds.csv",
)

display(cv_results)

In [ ]:
# @title 9.2. Сводка grouped CV

cv_summary = (
    cv_results
    .groupby("model")
    .agg(
        n_features=("n_features", "first"),
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
    )
    .reset_index()
    .sort_values("roc_auc_mean", ascending=False)
)

save_table(
    cv_summary,
    OUT["tables"] / "model_comparison_grouped_cv_summary.csv",
)

display(cv_summary)

In [ ]:
# @title 9.3. Визуализация ROC-AUC в grouped CV

fig, ax = plt.subplots(figsize=(9, 6))

models_order = cv_summary["model"].tolist()

data_to_plot = [
    cv_results[cv_results["model"] == model]["roc_auc"].to_numpy()
    for model in models_order
]

box = ax.boxplot(
    data_to_plot,
    labels=models_order,
    patch_artist=True,
    showfliers=False,
)

for patch in box["boxes"]:
    patch.set_alpha(1.0)

ax.set_title("ROC-AUC в grouped cross-validation")
ax.set_ylabel("ROC-AUC")
ax.set_ylim(0, 1.05)

figure_path = OUT["figures"] / "grouped_cv_roc_auc_boxplot.png"
save_figure(fig, figure_path)

plt.show()

# 10. Permutation tests

Permutation test проверяет, насколько качество модели выше случайного уровня при перемешивании меток.

Это особенно полезно для компактной модели, поскольку позволяет показать, что результат не является случайным следствием малого числа признаков.

In [ ]:
# @title 10.1. Permutation tests для основных моделей

def run_permutation_test(
    df: pd.DataFrame,
    selected_features: list[str],
    model_name: str,
) -> dict:
    """
    Выполняет permutation test с grouped CV.

    Parameters
    ----------
    df : pd.DataFrame
        Таблица признаков.
    selected_features : list[str]
        Список признаков.
    model_name : str
        Название модели.

    Returns
    -------
    dict
        Результаты permutation test.
    """
    X, y, groups = prepare_ml_data(df, selected_features)

    cv = StratifiedGroupKFold(
        n_splits=CONFIG["n_splits_cv"],
        shuffle=True,
        random_state=CONFIG["random_state"],
    )

    score, permutation_scores, p_value = permutation_test_score(
        estimator=make_logreg_pipeline(),
        X=X,
        y=y,
        groups=groups,
        cv=cv,
        scoring="roc_auc",
        n_permutations=CONFIG["n_permutations"],
        random_state=CONFIG["random_state"],
        n_jobs=-1,
    )

    return {
        "model": model_name,
        "n_features": len(selected_features),
        "roc_auc_score": score,
        "permutation_mean": float(np.mean(permutation_scores)),
        "permutation_std": float(np.std(permutation_scores)),
        "p_value": p_value,
    }


permutation_models = [
    model_name for model_name in ["bp_idx_baseline", "compact_event", "all_features"]
    if model_name in feature_sets
]

permutation_results = []

for model_name in permutation_models:
    result = run_permutation_test(
        df=all_features_record_level,
        selected_features=feature_sets[model_name],
        model_name=model_name,
    )
    permutation_results.append(result)

permutation_results = pd.DataFrame(permutation_results)

save_table(
    permutation_results,
    OUT["tables"] / "permutation_test_results.csv",
)

display(permutation_results)

# 11. Stress tests

В этом разделе выполняются дополнительные проверки устойчивости:

1. модель на случайных признаках;
2. модель с добавленным шумовым признаком;
3. сравнение с baseline.

Эти тесты не являются основным результатом, но помогают показать, что качество модели не возникает из-за технического артефакта.

In [ ]:
# @title 11.1. Noise feature test

rng = np.random.default_rng(CONFIG["random_state"])

stress_df = all_features_record_level.copy()
stress_df["noise__random_normal"] = rng.normal(size=len(stress_df))

stress_feature_sets = {
    "noise_only": ["noise__random_normal"],
}

if "compact_event" in feature_sets:
    stress_feature_sets["compact_event_plus_noise"] = (
        feature_sets["compact_event"] + ["noise__random_normal"]
    )

stress_results = []

for model_name, selected_features in stress_feature_sets.items():
    metrics, _ = fit_and_evaluate_holdout(
        df=stress_df,
        selected_features=selected_features,
        model_name=model_name,
        train_idx=train_idx,
        test_idx=test_idx,
    )
    stress_results.append(metrics)

stress_results = pd.DataFrame(stress_results)

save_table(
    stress_results,
    OUT["tables"] / "stress_test_noise_holdout.csv",
)

display(stress_results)

# 12. Интерпретируемость компактной событийной модели

Для логистической регрессии можно извлечь коэффициенты после стандартизации признаков.

Коэффициенты помогают понять, какие признаки вносят наибольший вклад в решение компактной модели.

Интерпретировать их нужно осторожно, особенно если признаки коррелированы.

In [ ]:
# @title 12.1. Коэффициенты компактной событийной модели

if "compact_event" in trained_models:
    compact_features = feature_sets["compact_event"]
    compact_model = trained_models["compact_event"]

    coefficients = compact_model.named_steps["model"].coef_[0]

    compact_coefficients = pd.DataFrame(
        {
            "feature": compact_features,
            "coefficient": coefficients,
            "abs_coefficient": np.abs(coefficients),
        }
    ).sort_values("abs_coefficient", ascending=False)

    save_table(
        compact_coefficients,
        OUT["tables"] / "compact_event_model_coefficients.csv",
    )

    display(compact_coefficients)

    fig, ax = plt.subplots(figsize=(8, 5))

    coeff_plot = compact_coefficients.sort_values(
        "abs_coefficient",
        ascending=True,
    )

    ax.barh(
        coeff_plot["feature"],
        coeff_plot["coefficient"],
    )
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title("Коэффициенты компактной событийной модели")
    ax.set_xlabel("Стандартизированный коэффициент")

    figure_path = OUT["figures"] / "compact_event_model_coefficients.png"
    save_figure(fig, figure_path)

    plt.show()
else:
    print("Компактная событийная модель не была обучена.")

# 13. Итоговая таблица моделей для ВКР

В финальную таблицу включаются основные модели и ключевые метрики.

Рекомендуется использовать эту таблицу в разделе результатов:

- baseline;
- all features;
- compact event model.

In [ ]:
# @title 13.1. Финальная таблица моделей

cv_for_thesis = cv_summary.copy()
cv_for_thesis = cv_for_thesis.rename(
    columns={
        "model": "Модель",
        "n_features": "Число признаков",
        "roc_auc_mean": "ROC-AUC, mean",
        "roc_auc_std": "ROC-AUC, SD",
        "balanced_accuracy_mean": "Balanced accuracy, mean",
        "balanced_accuracy_std": "Balanced accuracy, SD",
        "f1_mean": "F1, mean",
        "f1_std": "F1, SD",
    }
)

model_name_ru = {
    "spectral_baseline": "Спектральная baseline-модель",
    "bp_idx_baseline": "Bandpower + slow/fast baseline",
    "all_features": "Все признаки",
    "compact_event": "Компактная событийная модель",
}

cv_for_thesis["Модель"] = cv_for_thesis["Модель"].replace(model_name_ru)

save_table(
    cv_for_thesis,
    OUT["tables"] / "model_comparison_summary_for_thesis.csv",
)

display(cv_for_thesis)

In [ ]:
# @title 13.2. График сравнения моделей для ВКР

plot_df = cv_summary.copy().sort_values("roc_auc_mean", ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))

ax.barh(
    plot_df["model"].replace(model_name_ru),
    plot_df["roc_auc_mean"],
    xerr=plot_df["roc_auc_std"],
    capsize=4,
)

ax.set_title("Сравнение моделей по ROC-AUC")
ax.set_xlabel("ROC-AUC, grouped CV")
ax.set_xlim(0, 1.05)

figure_path = OUT["figures"] / "model_comparison_roc_auc_for_thesis.png"
save_figure(fig, figure_path)

plt.show()

# 14. Финальная проверка результатов

На последнем этапе проверяется наличие ключевых таблиц и рисунков.

In [ ]:
# @title 14.1. Финальная проверка выходных файлов

required_outputs = {
    "all_features_record_level_core": (
        OUT["tables"] / "all_features_record_level_core.csv"
    ),
    "subject_level_core": (
        OUT["tables"] / "all_features_subject_level_core.csv"
    ),
    "feature_audit": OUT["tables"] / "feature_audit.csv",
    "feature_statistics_subject_level": (
        OUT["tables"] / "feature_statistics_subject_level.csv"
    ),
    "model_comparison_holdout": (
        OUT["tables"] / "model_comparison_holdout.csv"
    ),
    "grouped_cv_summary": (
        OUT["tables"] / "model_comparison_grouped_cv_summary.csv"
    ),
    "permutation_test_results": (
        OUT["tables"] / "permutation_test_results.csv"
    ),
    "model_comparison_summary_for_thesis": (
        OUT["tables"] / "model_comparison_summary_for_thesis.csv"
    ),
}

print("Проверка ключевых выходных файлов")
print("-" * 70)

for name, path in required_outputs.items():
    status = "OK" if path.exists() else "НЕ НАЙДЕН"
    print(f"{name}: {status}")
    print(f"  {path}")

print("\nСтатистика и ML завершены.")

# Итог ноутбука

В результате выполнения ноутбука сформирована итоговая матрица признаков:

```text
all_features_record_level_core.csv
```

Также были рассчитаны:

- feature audit;
- subject-level таблица признаков;
- subject-level статистика;
- baseline-модель;
- модель на всех признаках;
- компактная событийная модель;
- permutation tests;
- stress tests;
- итоговая таблица моделей для ВКР.

При описании результатов важно подчёркивать не только качество классификации, но и контроль утечки данных за счёт subject-level разбиения.

# 15. Русские названия признаков и таблицы для ВКР

Этот раздел добавляет человекочитаемые русские подписи признаков и формирует компактные таблицы для текста ВКР.

Также здесь фиксируется точная компактная событийная модель на двух признаках:

```text
event__event_mean_z_peak
event__event_mean_propagation_ms
```

Эта модель соответствует интерпретации: снижение нормированной амплитуды событий и увеличение времени их распространения.

In [ ]:
# @title 15.1. Словарь русских названий признаков

FEATURE_RU_PATTERNS = {
    "rel_power_delta": "относительная δ-мощность",
    "rel_power_theta": "относительная θ-мощность",
    "rel_power_alpha": "относительная α-мощность",
    "rel_power_beta": "относительная β-мощность",
    "rel_power_gamma": "относительная γ-мощность",
    "idx_slow_alpha": "индекс (δ+θ)/α",
    "idx_slow_beta": "индекс (δ+θ)/β",
    "idx_log_slow_alpha_beta": "лог-индекс (δ+θ)/(α+β)",
    "sef50": "SEF50",
    "sef95": "SEF95",
    "spectral_entropy": "спектральная энтропия",
    "spectral_flatness": "spectral flatness",
    "aperiodic_exponent": "апериодический exponent",
    "aperiodic_offset": "апериодический offset",
    "alpha_peak_frequency": "частота α-пика",
    "alpha_peak_amplitude": "амплитуда α-пика",
    "dfa_exponent": "DFA-экспонента",
    "burst_rate_per_min": "частота burst-эпизодов",
    "burst_time_fraction": "доля времени в burst",
    "mean_strength": "средняя сила связности",
    "clustering": "кластеризация сети",
    "global_efficiency": "глобальная эффективность",
    "small_world_proxy": "показатель small-world",
    "event__n_events": "число событий",
    "event__event_rate_per_min": "частота событий",
    "event__event_mean_z_peak": "средняя z-амплитуда событий",
    "event__event_mean_propagation_ms": "среднее время распространения событий",
    "event__event_mean_active_roi": "среднее число активных ROI",
    "event__event_mean_active_channels": "среднее число активных каналов",
    "event__event_mean_line_length": "line length событий",
    "event__event_mean_area": "площадь событий",
}


ROI_RU = {
    "frontal": "лобная ROI",
    "central": "центральная ROI",
    "temporal": "височная ROI",
    "parietal": "теменная ROI",
    "occipital": "затылочная ROI",
}


def make_feature_ru_name(feature: str) -> str:
    """
    Формирует короткое русское название признака.

    Parameters
    ----------
    feature : str
        Техническое имя признака.

    Returns
    -------
    str
        Русское название.
    """
    if feature in FEATURE_RU_PATTERNS:
        return FEATURE_RU_PATTERNS[feature]

    parts = feature.split("__")

    if len(parts) >= 3:
        block, context, metric = parts[0], parts[1], "__".join(parts[2:])
        metric_ru = FEATURE_RU_PATTERNS.get(metric, metric)
        context_parts = context.split("_")
        roi_candidates = [
            item for item in context_parts
            if item in ROI_RU
        ]

        if roi_candidates:
            roi_ru = ROI_RU[roi_candidates[0]]
            return f"{metric_ru}, {roi_ru}"

        return f"{metric_ru}, {context}"

    return FEATURE_RU_PATTERNS.get(feature, feature)


feature_name_table = pd.DataFrame(
    {
        "feature": feature_columns,
        "feature_ru": [
            make_feature_ru_name(feature)
            for feature in feature_columns
        ],
    }
)

save_table(
    feature_name_table,
    OUT["tables"] / "feature_names_ru.csv",
)

display(feature_name_table.head(30))

In [ ]:
# @title 15.2. Top-10 признаков для ВКР

top10_features_for_thesis = (
    feature_statistics_subject
    .dropna(subset=["hedges_g"])
    .assign(abs_g=lambda df: df["hedges_g"].abs())
    .sort_values("abs_g", ascending=False)
    .head(10)
    .copy()
)

top10_features_for_thesis = top10_features_for_thesis.merge(
    feature_name_table,
    on="feature",
    how="left",
)

top10_features_for_thesis = top10_features_for_thesis[
    [
        "feature",
        "feature_ru",
        "block",
        "tbi_mean",
        "control_mean",
        "delta_tbi_minus_control",
        "hedges_g",
        "p_value",
        "q_fdr",
    ]
]

save_table(
    top10_features_for_thesis,
    OUT["tables"] / "top10_features_for_thesis.csv",
)

display(top10_features_for_thesis)

In [ ]:
# @title 15.3. Точная компактная событийная модель на двух признаках

compact_event_2_features = [
    "event__event_mean_z_peak",
    "event__event_mean_propagation_ms",
]

missing_compact_features = [
    feature for feature in compact_event_2_features
    if feature not in all_features_record_level.columns
]

if missing_compact_features:
    raise ValueError(
        "Не найдены признаки для 2-feature compact event model: "
        f"{missing_compact_features}. "
        "Проверьте, что выполнен 03_temporal_connectivity_events_v2."
    )

feature_sets["compact_event_2_features"] = compact_event_2_features

metrics_2f, model_2f = fit_and_evaluate_holdout(
    df=all_features_record_level,
    selected_features=compact_event_2_features,
    model_name="compact_event_2_features",
    train_idx=train_idx,
    test_idx=test_idx,
)

trained_models["compact_event_2_features"] = model_2f

compact_2f_holdout = pd.DataFrame([metrics_2f])

save_table(
    compact_2f_holdout,
    OUT["tables"] / "compact_event_2_features_holdout.csv",
)

display(compact_2f_holdout)

In [ ]:
# @title 15.4. Grouped CV для точной 2-feature compact event model

compact_2f_cv = cross_validate_grouped(
    df=all_features_record_level,
    selected_features=compact_event_2_features,
    model_name="compact_event_2_features",
)

compact_2f_cv_summary = (
    compact_2f_cv
    .groupby("model")
    .agg(
        n_features=("n_features", "first"),
        roc_auc_mean=("roc_auc", "mean"),
        roc_auc_std=("roc_auc", "std"),
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_std=("balanced_accuracy", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
    )
    .reset_index()
)

save_table(
    compact_2f_cv,
    OUT["tables"] / "compact_event_2_features_cv_folds.csv",
)

save_table(
    compact_2f_cv_summary,
    OUT["tables"] / "compact_event_2_features_cv_summary.csv",
)

display(compact_2f_cv_summary)

In [ ]:
# @title 15.5. Обновлённая итоговая таблица моделей для ВКР

model_comparison_summary_for_thesis_v2 = pd.concat(
    [
        cv_summary,
        compact_2f_cv_summary,
    ],
    ignore_index=True,
)

model_name_ru_v2 = {
    "spectral_baseline": "Спектральная baseline-модель",
    "bp_idx_baseline": "Bandpower + slow/fast baseline",
    "all_features": "Все признаки",
    "compact_event": "Компактная событийная модель",
    "compact_event_2_features": "Компактная событийная модель, 2 признака",
}

model_comparison_summary_for_thesis_v2["model_ru"] = (
    model_comparison_summary_for_thesis_v2["model"]
    .replace(model_name_ru_v2)
)

save_table(
    model_comparison_summary_for_thesis_v2,
    OUT["tables"] / "model_comparison_summary_for_thesis_v2.csv",
)

display(model_comparison_summary_for_thesis_v2)